In [1]:
!git clone https://github.com/Fowjia-Peta/alzheimers-mri-detection.git
%cd /content/alzheimers-mri-detection
!pip install -q -r requirements.txt kagglehub
import kagglehub, os, shutil, random
p = kagglehub.dataset_download("pashacqu/alzheimers-dataset-4-class-of-images")
src = None
for r, d, f in os.walk(p):
    if os.path.basename(r) == "OriginalDataset":
        src = r
        break
print(src)
random.seed(42)
for cls in sorted(os.listdir(src)):
    files = sorted(os.listdir(os.path.join(src, cls)))
    random.shuffle(files)
    n = int(0.8 * len(files))
    for split, fl in (("train", files[:n]), ("test", files[n:])):
        out = os.path.join("dataset", split, cls)
        os.makedirs(out, exist_ok=True)
        for fn in fl:
            shutil.copy(os.path.join(src, cls, fn), out)
print("done")

Cloning into 'alzheimers-mri-detection'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 27 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 14.39 KiB | 254.00 KiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/alzheimers-mri-detection
Using Colab cache for faster access to the 'alzheimers-dataset-4-class-of-images' dataset.
/kaggle/input/alzheimers-dataset-4-class-of-images/OriginalDataset
done


In [2]:
!find dataset -type f | wc -l
!find dataset/train -type f | wc -l
!find dataset/test -type f | wc -l
!nvidia-smi

6400
5119
1281
Sat Nov  7 11:25:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------------

In [3]:
!python train.py --data dataset --epochs 20

I0000 00:00:1794052163.192339    8743 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 5119 files belonging to 4 classes.
W0000 00:00:1794052168.863759    8743 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1794052168.866614    8743 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Found 1281 files belonging to 4 classes.
Epoch 1/20
I0000 00:00:1794052176.375136    8792 service.cc:153] XLA service 0x7e13a0009d80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:179405217

In [4]:
!cat saved_models/results.json

{
  "cnn": 0.9664324522018433,
  "mobilenetv2": 0.8430913090705872,
  "vgg16": 0.7189695835113525
}

In [6]:
!sed -i 's/fit(vgg, train_ds, val_ds, 5, cw)/fit(vgg, train_ds, val_ds, args.epochs, cw)/' train.py
!sed -i 's/    os.makedirs(args.out, exist_ok=True)/    tf.keras.utils.set_random_seed(42)\n    os.makedirs(args.out, exist_ok=True)/' train.py
!grep -n "args.epochs\|set_random_seed" train.py

52:    tf.keras.utils.set_random_seed(42)
62:    fit(cnn, train_ds, val_ds, args.epochs, cw)
69:    fit(mn, train_ds, val_ds, args.epochs, cw)
77:    fit(mn, train_ds, val_ds, args.epochs, cw)
84:    fit(vgg, train_ds, val_ds, args.epochs, cw)
89:    fit(vgg, train_ds, val_ds, args.epochs, cw)


In [7]:
!python train.py --data dataset --epochs 10 2>&1 | tee saved_models/training_log.txt

I0000 00:00:1794053683.894387   20468 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 5119 files belonging to 4 classes.
W0000 00:00:1794053689.649164   20468 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1794053689.651445   20468 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Found 1281 files belonging to 4 classes.
Epoch 1/10
I0000 00:00:1794053696.312676   20515 service.cc:153] XLA service 0x7c8e28005500 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:179405369

In [8]:
import tensorflow as tf, numpy as np
from sklearn.metrics import confusion_matrix
val = tf.keras.utils.image_dataset_from_directory(
    "dataset/test", image_size=(128,128), batch_size=32,
    label_mode="int", shuffle=False)
m = tf.keras.models.load_model("saved_models/cnn.keras")
y = np.concatenate([l.numpy() for _, l in val])
p = m.predict(val).argmax(1)
print(val.class_names)
print(confusion_matrix(y, p))
print("true counts:", np.bincount(y))
print("pred counts:", np.bincount(p, minlength=4))

Found 1281 files belonging to 4 classes.
41/41 ━━━━━━━━━━━━━━━━━━━━ 5s 66ms/step
['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
[[ 16 162   0   2]
 [  0  13   0   0]
 [ 28 595   0  17]
 [ 16 423   0   9]]
true counts: [180  13 640 448]
pred counts: [  60 1193    0   28]


In [10]:
!sed -i 's/float(w) for i, w in enumerate(weights)/float(np.sqrt(w)) for i, w in enumerate(weights)/' train.py
!grep -n "np.sqrt" train.py

30:    class_weight = {int(i): float(np.sqrt(w)) for i, w in enumerate(weights)}


In [11]:
!python train.py --data dataset --epochs 10 2>&1 | tee saved_models/training_log.txt

Found 5119 files belonging to 4 classes.
W0000 00:00:1794054676.412518   29829 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
Found 1281 files belonging to 4 classes.
Epoch 1/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 26s 104ms/step - accuracy: 0.4821 - loss: 1.0655 - val_accuracy: 0.5316 - val_loss: 0.9247
Epoch 2/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.5526 - loss: 0.9136 - val_accuracy: 0.6042 - val_loss: 0.8229
Epoch 3/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.6267 - loss: 0.7408 - val_accuracy: 0.7143 - val_loss: 0.6819
Epoch 4/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.7205 - loss: 0.5409 - val_accuracy: 0.7970 - val_loss: 0.4749
Epoch 5/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.8105 - loss: 0.3696 - val_accuracy: 0.8212 - val_loss: 0.4180
Epoch 6/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.8781 - loss

In [12]:
!cat saved_models/results.json

{
  "cnn": 0.9695550203323364,
  "mobilenetv2": 0.8899297714233398,
  "vgg16": 0.9180327653884888
}

In [17]:
import tensorflow as tf, numpy as np
from sklearn.metrics import confusion_matrix
val = tf.keras.utils.image_dataset_from_directory(
    "dataset/test", image_size=(128,128), batch_size=32,
    label_mode="int", shuffle=False)
m = tf.keras.models.load_model("saved_models/cnn.keras")
y = np.concatenate([l.numpy() for _, l in val])
p = m.predict(val).argmax(1)
print(val.class_names)
print(confusion_matrix(y, p))
print("true counts:", np.bincount(y))
print("pred counts:", np.bincount(p, minlength=4))

Found 1281 files belonging to 4 classes.
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step
['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
[[175   0   1   4]
 [  0  13   0   0]
 [  2   0 620  18]
 [  2   0  12 434]]
true counts: [180  13 640 448]
pred counts: [179  13 633 456]
